In [ ]:
import numpy as np
import pandas as pd
import datetime
import tqdm
import os
import sys
import itertools

In [ ]:
from google.colab import files
upload = files.upload()

In [ ]:
##Prelab activities 
disease_name = ['ALS', 'Ovarian_Cancer', 'Parkinson_Disease', 'Schizophrenia'] #list of disease names
info_dataframe = pd.read_csv('gene_information.csv', sep = ',', header = 0) #read content in gene_information.csv and return a dataframe info_dataframe
gene_dictionary = {}  #initialize empty dictionary (keys: gene names or synonyms, values: gene IDs)

for _, row in info_dataframe.iterrows(): #iterates through each row of info_dataframe
    if pd.notnull(row[1]):
        gene_dictionary[row[1]] = row[0] #assign key: gene name (from column 2), value: gene ID (column 1) if the second column of row isn't empty
    if str(row[2]) == '-' : 
      continue
    if pd.notnull(row[2]):
        for gene in str(row[2]).split('|') : #split the entries (separated by |) if the third column isn't empty
          gene_dictionary[row[2]] = row[0] #assign key: each synonym (from column 3), value: gene ID (column 1)

#create gene tagging table
for disease in disease_name:
    literature_file = disease + '_abstract.txt' #construct a literature file
    with open(literature_file) as file:
        rows = [] #initialize list rows
        PMID = -1
        count = 0
        for line in file:
            count += 1
            if count % 100 == 0: 
              print("line", count) #check the progress
            if 'PMID' == line[0:4]:
                PMID = str(line.split(':')[1]).strip() #extract PMID value if the line starts with string "PMID"
                continue
            for sent_id, sentence in enumerate(line.split('. ')) : #split each line into sentences (when ". " present) and iterates over each sentence
                for gene in gene_dictionary: 
                  sent_len = len(sentence.split(' '))
                  for sent_pos, word in enumerate(sentence.split(' ')) : 
                    if gene in word : 
                      rows.append({
                          "GeneID": gene_dictionary[gene], 
                          "PMID": PMID, 
                          "SentenceID": sent_id, 
                          "SentenceLength": sent_len, 
                          "GeneLocation": sent_pos, 
                      }) #create a row with gene-related information and this row is appended to list rows if any gene in gene_dictionary is present in sentence
                
        df = pd.DataFrame(rows) #generate dataframe df using list rows
    print(disease, "DONE!") #check the progress
    df.to_csv(disease + '_gene_table.txt') #save a dataframe as <DiseaseName>_gene_table.txt file

#create disease tagging table
syn = [] #initialize empty list syn
for disease in disease_name: #iterate over each diseas in disease_name list
  syn.append({disease: disease}) #create dictionary entry with disease name as key and value, and append this entry to list syn
  with open(disease + '_synonyms.txt') as file : 
    for l in file : 
      syn.append({l.strip(): disease}) #create a dictionary entry with key: stripped line, value: disease name

for disease in disease_name:
  literature_file = disease + '_abstract.txt' #create a literature file
  PMID = -1
  with open(literature_file) as file:
    rows = [] #initialize list rows
    PMID = -1 #initialize PMID value to -1
    count = 0
    for line in file : 
      count += 1 #increment variable count
      if count % 100 == 0 : 
        print("line", count) #check the progress
      if "PMID" == line[0:4] : 
        PMID = str(line.split(':')[1]).strip() #extract PMID value if the line starts with string "PMID"
        continue
      for sent_id, sentence in enumerate(line.split('. ')): #split each line into sentences (when ". " present) and iterates over each sentence
        words = sentence.split(' ') #split words for each sentence
        for dis in syn : 
          for dis_name in dis : #iterate over disease synonyms in list syn
            dis_syn = dis[dis_name]
            sent_len = len(words)
            dis_token = dis_name.split(' ')
            for sent_pos, word in enumerate(words) : 
              for i in range(len(dis_token)) : 
                if dis_token[i] != words[sent_pos + i] : #compare each word with disease name or synonym
                  break
                if i == len(dis_token) - 1: #if the words match
                  rows.append({
                      'DiseaseName': dis_syn, 
                      'PMID': PMID, 
                      'SentenceID': sent_id, 
                      'SentenceLength': sent_len,
                      'DiseaseLocation': sent_pos,
                  }) #create a row with disease-related information and this row is appended to list rows if any disease name or synonym is present in sentence
      
      df = pd.DataFrame(rows) #generate dataframe df using list rows
    print("DONE", disease) #check the progress
    df.to_csv(disease + '_disease_table.txt') #save a dataframe as <DiseaseName>_disease_table.txt file

In [ ]:
def load_files(disease_name: str):
  gene_table = pd.read_csv('gene_information.csv', sep = ',', header = 0) #read contents in gene_information.csv and return a dataframe of gene_table
  synonym_dataframe = pd.read_csv(disease_name + '_synonyms.txt', sep = '\t', header = None) #read contents in <DiseaseName>_synonyms.txt and return a dataframe of synonym_dataframe
  disease_synonym_list = list(synonym_dataframe.iloc[0]) #return a list of disease synonyms
  disease_literature = disease_name + '_abstract.txt' #return disease literature filename
  return gene_table, disease_synonym_list, disease_literature

In [ ]:
def count_disease_occurrence(disease_name: str, disease_literature, disease_synonym_list):
    disease_occurrence = pd.read_csv(disease_name + '_disease_table.txt', sep = ',', index_col = 0) #read the file <Disease Name>_disease_table.txt and return disease_occurrence dataframe
    disease_occurrence.columns = ['DiseaseName', 'PMID', 'SentenceID', 'SentenceLength', 'DiseaseLocation'] #dataframe column labels
    return disease_occurrence

In [ ]:
def count_gene_occurrence(disease_name: str, disease_literature, gene_table):
    gene_occurrence = pd.read_csv(disease_name + '_gene_table.txt', sep = ',', index_col = 0) #read the file <Disease Name>_gene_table.txt and return gene_occurrence dataframe
    gene_occurrence.columns = ['GeneID', 'PMID', 'SentenceID', 'SentenceLength', 'GeneLocation'] #dataframe column labels
    return gene_occurrence

In [ ]:
def scoring_function(*args, **kwargs):
    return score

In [ ]:
def scoring_function_2(*args, **kwargs):
    return score

In [ ]:
def score_sentences(disease_occurrence, gene_occurrence, scoring_hyperparameters):
  gene_score_dict = {} #initialize empty dictionary
  gene_coefficient = {} #initialize empty dictionary
  disease_loc_dict = {} #initialize empty dictionary
  
  for row in gene_occurrence.iterrows(): #iterate over each row of gene_occurence
    if row[1][0] in gene_score_dict:
      gene_coefficient[(row[1][0], row[1][1])] += 1 #increment count for the combination of gene name and PMID (occurrence) if the gene name is already in gene_score_dict
    else:
      gene_coefficient[(row[1][0], row[1][1])] = 1 #set count to 1 if gene name isn't in gene_score_dict
  
  for row in disease_occurrence.iterrows(): #iterate over each row in disease_occurence dataframe
    disease_loc_dict[(row[1][1], row[1][2])] = row[1][4] #assign key: PMID-sentence ID tuple, value: disease location to disease_occurrence dictionary
  for row in gene_occurrence.iterrows():
    if (row[1][1], row[1][2]) in disease_loc_dict: #check whether PMID-sentence ID combination is in disease_occurrence_dict
      if row[1][0] in gene_score_dict:
        gene_score_dict[row[1][0]] += gene_coefficient[(row[1][0], row[1][1])]/(1 + abs(row[1][4] - disease_loc_dict[(row[1][1], row[1][2])]))
        #increment the score by gene name-PMID combination count / (1 + gene-disease location absolute difference) if the gene name is in gene_score_dict
      else:
        gene_score_dict[row[1][0]] = gene_coefficient[(row[1][0], row[1][1])]/(1 + abs(row[1][4] - disease_loc_dict[(row[1][1], row[1][2])]))
        #assign the score if gene name isn't in gene_score_dict
        
  return gene_score_dict #return gene_score_dict with keys: gene names, values: scores

In [ ]:
def print_final_gene_list(gene_score_list, print_threshold):
    gene_list_final = [str(_gene) for _gene in gene_score_list if gene_score_list[_gene] > print_threshold]
    gene_list_print_string = ','.join(gene_list_final)
    print(gene_list_print_string)  # print gene list in single line, seperated by comma.
    return gene_list_print_string

In [ ]:
def calculate_f1_score(disease, string_to_check):
    filename = disease + '_targets.txt'
    with open(filename, 'r') as GS_file:
        GS_list = GS_file.readlines()
    new_list = []
    for each in GS_list:
        new_item = each.strip()
        new_list.append(new_item)
    test_gene_list = list(np.unique(string_to_check.split(',')))

    TP, FP = 0, 0
    FN = len(new_list)
    for gene in test_gene_list:
        if gene in new_list:
            TP += 1
            FN -= 1
        else:
            FP += 1
    precision = TP / (TP + FP)
    recall = TP / (TP + FN)
    try:
        f1_score = 2 * precision * recall / (precision + recall)
    except ZeroDivisionError:
        f1_score = 0
    print("Precision and recall is: ", precision, ",", recall)
    print("F1 score is: ", f1_score)
    return None

In [ ]:
def main(disease, mode='run'):
    threshold = 1  # mutable attribute
    hyperparameters = [0, 1, 2, 3, 4]  # mutable attribute, MUST be list for expandability.
    # below structure is immutable.
    run_start_time = datetime.datetime.utcnow()
    gene_table_data, disease_synonym_list_data, disease_literature_data = load_files(disease)
    disease_occurrence_data = count_disease_occurrence(disease, disease_literature_data, disease_synonym_list_data)
    gene_occurrence_data = count_gene_occurrence(disease, disease_literature_data, gene_table_data)
    disease_related_genes_dictionary = score_sentences(disease_occurrence_data, gene_occurrence_data, hyperparameters)
    final_gene_string = print_final_gene_list(disease_related_genes_dictionary, threshold)
    run_end_time = datetime.datetime.utcnow()
    total_runtime = run_end_time - run_start_time
    print("Total runtime:", total_runtime.total_seconds(), 'seconds')
    if mode == 'check':
        print(disease, final_gene_string)
        calculate_f1_score(disease, final_gene_string)
    else:
        pass
    return None

In [ ]:
disease = "ALS"
threshold = 1  # mutable attribute
hyperparameters = [0, 1, 2, 3, 4]  # mutable attribute, MUST be list for expandability.
# below structure is immutable.
run_start_time = datetime.datetime.utcnow()
gene_table_data, disease_synonym_list_data, disease_literature_data = load_files(disease)
disease_occurrence_data = count_disease_occurrence(disease, disease_literature_data, disease_synonym_list_data)
gene_occurrence_data = count_gene_occurrence(disease, disease_literature_data, gene_table_data)
disease_related_genes_dictionary = score_sentences(disease_occurrence_data, gene_occurrence_data, hyperparameters)
final_gene_string = print_final_gene_list(disease_related_genes_dictionary, threshold)
run_end_time = datetime.datetime.utcnow()
total_runtime = run_end_time - run_start_time
print("Total runtime:", total_runtime.total_seconds(), 'seconds')

In [ ]:
main('ALS', mode='run') # Amyotropic Lateral Sclerosis
main('Schizophrenia', mode='run') # Schizophrenia
main('Parkinson_Disease', mode='run') # Parkinson's Disease

In [ ]:
main('Ovarian_Cancer') # Ovarian Cancer